In [72]:
# Import dependencies
import numpy as np
import pandas as pd
import statsmodels.stats.power as smp
import scipy.stats as stats
from statsmodels.stats.proportion import proportions_ztest
import statsmodels.formula.api as sm
import math

# Overview
- This notebook corresponds with the Experiments Module Review lecture, in which we run through the entire experimental design process using a mock survey experiment. Below are...
1. Minimum sample size calculations (pre-analysis plan)
2. Data processing and description (including balance tables)
3. Analysis of test results (3 treatments)
4. Statistical power analysis

## 1. Minimal Sample Size Calculation
- Need other 3 parameters: effect size (Cohen's h), significance level (alpha), statistical power (beta)

In [73]:
## Calculate minimum sample size with statsmodels

# Define parameters 
effect_size = 0.2 # Cohen's h for small effect
alpha = 0.05 # Standard significance level
power = 0.80 # Standard statistical power

# Calculate minimum sample size *per group*
min_n1 = smp.NormalIndPower().solve_power(effect_size=effect_size, alpha=alpha, power=power, 
                                          nobs1=None)

print(min_n1)

392.4430232577885


## 2. Data Processing and Description
- Loading in data and renaming columns
- Data engineering: treatment conditions and outcome measures
- Balance tables

In [74]:
## Load in

data_df = pd.read_csv('MDining_survey_experiment_data.csv')
data_df.describe(include='all').T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
ResponseId,23,23,R_1jkQFD7koam1P4C,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Duration (in seconds),23.0,NaN,NaN,NaN,18533.391304,88704.057274,15.0,22.0,33.0,39.0,425447.0
Q1,23,1,"I consent, begin the study",23,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Q187,23,2,18-24 years old,13,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Q188,23,4,Female,15,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Q184,21,6,Asian,8,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Q183,23,2,No,19,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Q17,5,2,Somewhat bad,3,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Q17_DO,5,2,Very good|Somewhat good|Neither good nor bad|S...,3,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Q18_1,6.0,NaN,NaN,NaN,63.333333,7.527727,50.0,61.25,65.0,68.75,70.0


In [75]:
# Rename columns
column_names = {'Q1':'consent',
                'Q187':'age_group',
                'Q188':'gender',
                'Q184':'race',
                'Q183':'hispanic_latino',
                'Q17':'product_likert',
                'Q17_DO':'product_likert_option_order',
                'Q18_1':'product_sliding',
                'Q193':'people_likert',
                'Q193_DO':'people_likert_option_order',
                'Q194_1':'people_sliding'}

data_df = data_df.rename(columns=column_names)
data_df.head()

,ResponseId,Duration (in seconds),consent,age_group,gender,race,hispanic_latino,product_likert,product_likert_option_order,product_sliding,people_likert,people_likert_option_order,people_sliding
0,R_1jkQFD7koam1P4C,18,"I consent, begin the study",25-34 years old,Male,Asian,Yes,NaN,NaN,NaN,Very good,Very good|Somewhat good|Neither good nor bad|S...,NaN
1,R_1ziP1dCyikYE6z9,15,"I consent, begin the study",18-24 years old,Male,White or Caucasian,No,NaN,NaN,NaN,Somewhat good,Very good|Somewhat good|Neither good nor bad|S...,NaN
2,R_7rDq5lKszYyOG5a,50,"I consent, begin the study",18-24 years old,Female,White or Caucasian,No,Somewhat bad,Very bad|Somewhat bad|Neither good nor bad|Som...,NaN,NaN,NaN,NaN
3,R_39QshBWHBf6Sgw1,43,"I consent, begin the study",18-24 years old,Female,NaN,Yes,NaN,NaN,50.0,NaN,NaN,NaN
4,R_1atrUlghoPTYPc1,37,"I consent, begin the study",18-24 years old,Female,Black or African American,No,NaN,NaN,NaN,NaN,NaN,75.0


In [76]:
## Data engineering

## Define treatment groups
data_df['question_framing'] = np.where((data_df['people_likert'].isna()) & (data_df['people_sliding'].isna()), 0, 1)
data_df['answer_format'] = np.where((data_df['product_likert'].notna()) | (data_df['people_likert'].notna()), 0, 1)
data_df['answer_option_order'] = np.NaN
data_df.loc[(data_df['product_likert_option_order'] == 'Very good|Somewhat good|Neither good nor bad|Somewhat bad|Very bad') | (data_df['people_likert_option_order'] == 'Very good|Somewhat good|Neither good nor bad|Somewhat bad|Very bad'),
            'answer_option_order'] = 1
data_df.loc[(data_df['product_likert_option_order'] == 'Very bad|Somewhat bad|Neither good nor bad|Somewhat good|Very good') | (data_df['people_likert_option_order'] == 'Very bad|Somewhat bad|Neither good nor bad|Somewhat good|Very good'),
            'answer_option_order'] = 0

## Define outcomes: convert likert and sliding to binary, 1=positive, 0=not positive
data_df['satisfaction'] = 0

# Likert, Somewhat good or Very good = 1
data_df.loc[(data_df['product_likert']=='Somewhat good') | (data_df['product_likert']=='Very good') | (data_df['people_likert']=='Somewhat good') | (data_df['people_likert']=='Very good') , 
            'satisfaction'] = 1

# Sliding, >50 = 1
data_df.loc[(data_df['product_sliding']>50) | (data_df['people_sliding']>50), 
            'satisfaction'] = 1

data_df.describe(include='all').T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
ResponseId,23,23,R_1jkQFD7koam1P4C,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Duration (in seconds),23.0,NaN,NaN,NaN,18533.391304,88704.057274,15.0,22.0,33.0,39.0,425447.0
consent,23,1,"I consent, begin the study",23,NaN,NaN,NaN,NaN,NaN,NaN,NaN
age_group,23,2,18-24 years old,13,NaN,NaN,NaN,NaN,NaN,NaN,NaN
gender,23,4,Female,15,NaN,NaN,NaN,NaN,NaN,NaN,NaN
race,21,6,Asian,8,NaN,NaN,NaN,NaN,NaN,NaN,NaN
hispanic_latino,23,2,No,19,NaN,NaN,NaN,NaN,NaN,NaN,NaN
product_likert,5,2,Somewhat bad,3,NaN,NaN,NaN,NaN,NaN,NaN,NaN
product_likert_option_order,5,2,Very good|Somewhat good|Neither good nor bad|S...,3,NaN,NaN,NaN,NaN,NaN,NaN,NaN
product_sliding,6.0,NaN,NaN,NaN,63.333333,7.527727,50.0,61.25,65.0,68.75,70.0


In [77]:
## Balance Tables (ideally running z- or t-tests for each measure)

# Question Framing Treatment
balance_table1 = data_df.groupby('question_framing').agg({
    'Duration (in seconds)': ['mean', 'std'],
    'age_group': lambda x: x.value_counts(normalize=True).round(4).to_dict(),
    'race': lambda x: x.value_counts(normalize=True).round(4).to_dict(),
    'hispanic_latino': lambda x: x.value_counts(normalize=True).round(4).to_dict()
}).reset_index()

print("Balance Table:")
for column in balance_table1.columns:
    print(balance_table1[column])

# Answer Format Treatment
balance_table2 = data_df.groupby('answer_format').agg({
    'Duration (in seconds)': ['mean', 'std'],
    'age_group': lambda x: x.value_counts(normalize=True).round(4).to_dict(),
    'race': lambda x: x.value_counts(normalize=True).round(4).to_dict(),
    'hispanic_latino': lambda x: x.value_counts(normalize=True).round(4).to_dict()
}).reset_index()

print("Balance Table:")
for column in balance_table2.columns:
    print(balance_table2[column])

# Answer Option Order Treatment
balance_table3 = data_df.groupby('answer_option_order').agg({
    'Duration (in seconds)': ['mean', 'std'],
    'age_group': lambda x: x.value_counts(normalize=True).round(4).to_dict(),
    'race': lambda x: x.value_counts(normalize=True).round(4).to_dict(),
    'hispanic_latino': lambda x: x.value_counts(normalize=True).round(4).to_dict()
}).reset_index()

print("Balance Table:")
for column in balance_table3.columns:
    print(balance_table3[column])


Balance Table:
0    0
1    1
Name: (question_framing, ), dtype: int64
0    38705.545455
1       42.250000
Name: (Duration (in seconds), mean), dtype: float64
0    128267.629906
1        47.443986
Name: (Duration (in seconds), std), dtype: float64
0    {'18-24 years old': 0.5455, '25-34 years old':...
1    {'18-24 years old': 0.5833, '25-34 years old':...
Name: (age_group, <lambda>), dtype: object
0    {'White or Caucasian': 0.5, 'Asian': 0.3, 'Whi...
1    {'Asian': 0.4545, 'White or Caucasian': 0.2727...
Name: (race, <lambda>), dtype: object
0    {'No': 0.9091, 'Yes': 0.0909}
1        {'No': 0.75, 'Yes': 0.25}
Name: (hispanic_latino, <lambda>), dtype: object
Balance Table:
0    0
1    1
Name: (answer_format, ), dtype: int64
0       27.083333
1    38722.090909
Name: (Duration (in seconds), mean), dtype: float64
0        10.299676
1    128262.150769
Name: (Duration (in seconds), std), dtype: float64
0    {'25-34 years old': 0.5833, '18-24 years old':...
1    {'18-24 years old': 0.7273, '

## 3. Analysis of Test Results
- Two-proportion z-tests

In [78]:
## Question Framing Treatment

satisfaction_rates1 = data_df.groupby('question_framing')['satisfaction'].mean()
print("Satisfaction Rate by Question Frmaing:")
print(satisfaction_rates1) # 1 = people-framing

# Get proportions and difference
proportion0 = satisfaction_rates1[0]
proportion1 = satisfaction_rates1[1]
diff_in_props = proportion1 - proportion0
print(diff_in_props)

# Get group sample sizes
n0 = data_df[data_df['question_framing'] == 0].shape[0]
n1 = data_df[data_df['question_framing'] == 1].shape[0]

# Get completion counts for each group
completion_count0 = data_df[data_df['question_framing'] == 0]['satisfaction'].sum()
completion_count1 = data_df[data_df['question_framing'] == 1]['satisfaction'].sum()

# Get pooled proportion
pooled_prop = (completion_count0 + completion_count1)/(n0 + n1)

# Get standard error
std_error = np.sqrt(pooled_prop * (1 - pooled_prop) * (1/n0 + 1/n1))

# Calculate z-statistic and confidence interval
z_stat = diff_in_props / std_error
print(f"Z-statistic: {z_stat}")

# Calculate p-value
p_value = 2 * (1 - stats.norm.cdf(abs(z_stat)))
print(f"P-value: {p_value}")

# Calculate 95% confidence interval
threshold_95 = 1.96
ci_lower = diff_in_props - threshold_95 * std_error
ci_upper = diff_in_props + threshold_95 * std_error
print(f"95% Confidence Interval: ({ci_lower}, {ci_upper})")

Satisfaction Rate by Question Frmaing:
question_framing
0    0.636364
1    0.833333
Name: satisfaction, dtype: float64
0.19696969696969702
Z-statistic: 1.0746083590645297
P-value: 0.2825501078088375
95% Confidence Interval: (-0.16228733169950155, 0.5562267256388955)


In [79]:
## Answer Format Treatment
satisfaction_rates2 = data_df.groupby('answer_format')['satisfaction'].mean()
print("Satisfaction Rate by Answer Format:")
print(satisfaction_rates2) # 1 = Likert scale

# Get proportions and difference
proportion0 = satisfaction_rates2[0]
proportion1 = satisfaction_rates2[1]
diff_in_props = proportion1 - proportion0
print(diff_in_props)

# Get group sample sizes
n0 = data_df[data_df['answer_format'] == 0].shape[0]
n1 = data_df[data_df['answer_format'] == 1].shape[0]

# Get completion counts for each group
completion_count0 = data_df[data_df['answer_format'] == 0]['satisfaction'].sum()
completion_count1 = data_df[data_df['answer_format'] == 1]['satisfaction'].sum()

# Get pooled proportion
pooled_prop = (completion_count0 + completion_count1)/(n0 + n1)

# Get standard error
std_error = np.sqrt(pooled_prop * (1 - pooled_prop) * (1/n0 + 1/n1))

# Calculate z-statistic and confidence interval
z_stat = diff_in_props / std_error
print(f"Z-statistic: {z_stat}")

# Calculate p-value
p_value = 2 * (1 - stats.norm.cdf(abs(z_stat)))
print(f"P-value: {p_value}")

# Calculate 95% confidence interval
threshold_95 = 1.96
ci_lower = diff_in_props - threshold_95 * std_error
ci_upper = diff_in_props + threshold_95 * std_error
print(f"95% Confidence Interval: ({ci_lower}, {ci_upper})")

Satisfaction Rate by Answer Format:
answer_format
0    0.666667
1    0.818182
Name: satisfaction, dtype: float64
0.1515151515151516
Z-statistic: 0.826621814665023
P-value: 0.4084514489299582
95% Confidence Interval: (-0.20774187715404696, 0.5107721801843501)


In [80]:
## Answer Option Order Treatment
satisfaction_rates3 = data_df.groupby('answer_option_order')['satisfaction'].mean()
print("Satisfaction Rate by Answer Option Order:")
print(satisfaction_rates3) # 1 = Positive first

# Get proportions and difference
proportion0 = satisfaction_rates3[0]
proportion1 = satisfaction_rates3[1]
diff_in_props = proportion1 - proportion0
print(diff_in_props)

# Get group sample sizes
n0 = data_df[data_df['answer_option_order'] == 0].shape[0]
n1 = data_df[data_df['answer_option_order'] == 1].shape[0]

# Get completion counts for each group
completion_count0 = data_df[data_df['answer_option_order'] == 0]['satisfaction'].sum()
completion_count1 = data_df[data_df['answer_option_order'] == 1]['satisfaction'].sum()

# Get pooled proportion
pooled_prop = (completion_count0 + completion_count1)/(n0 + n1)

# Get standard error
std_error = np.sqrt(pooled_prop * (1 - pooled_prop) * (1/n0 + 1/n1))

# Calculate z-statistic and confidence interval
z_stat = diff_in_props / std_error
print(f"Z-statistic: {z_stat}")

# Calculate p-value
p_value = 2 * (1 - stats.norm.cdf(abs(z_stat)))
print(f"P-value: {p_value}")

# Calculate 95% confidence interval
threshold_95 = 1.96
ci_lower = diff_in_props - threshold_95 * std_error
ci_upper = diff_in_props + threshold_95 * std_error
print(f"95% Confidence Interval: ({ci_lower}, {ci_upper})")

Satisfaction Rate by Answer Option Order:
answer_option_order
0.0    0.666667
1.0    0.666667
Name: satisfaction, dtype: float64
0.0
Z-statistic: 0.0
P-value: 1.0
95% Confidence Interval: (-0.6159685738336147, 0.6159685738336147)


4. Statistical Power Analysis
- Using actual sample and effect sizes from data

In [81]:
## Question Framing Treatment

# Define parameters 
effect_size = np.arcsin(math.sqrt(satisfaction_rates1[0])) - np.arcsin(math.sqrt(satisfaction_rates1[1])) # Cohen's h for small effect
print("Question Framing Effect in Cohen's h:", effect_size)
alpha = 0.05 # Standard significance level
nobs1 = 12

# Calculate minimum sample size *per group*
power_1 = smp.NormalIndPower().solve_power(effect_size=effect_size, alpha=alpha, power=None, 
                                          nobs1=nobs1)

print('Stat Power:', power_1) # 8% Power :(

Question Framing Effect in Cohen's h: -0.22675051273168756
Stat Power: 0.08602400495256676


In [82]:
## Answer Format Treatment

# Define parameters 
effect_size = np.arcsin(math.sqrt(satisfaction_rates2[0])) - np.arcsin(math.sqrt(satisfaction_rates2[1])) # Cohen's h for small effect
print("Answer Format Effect in Cohen's h:", effect_size)
alpha = 0.05 # Standard significance level
nobs1 = 12

# Calculate minimum sample size *per group*
power_2 = smp.NormalIndPower().solve_power(effect_size=effect_size, alpha=alpha, power=None, 
                                          nobs1=nobs1)

print('Stat Power:', power_2) # 7% Power :(

Answer Format Effect in Cohen's h: -0.17496904566568883
Stat Power: 0.07129539228770831
